# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# metadata object from mlcroissant has direct attributes, not subscriptable
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print("\nDataset identifier:", dataset.metadata.identifier)
print("\nAvailable record sets (@id):")

# List all record sets with their @id and name
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '<unnamed>')}")

## 2. Data Overview
Review available record sets, as well as their fields and columns using their `@id` values.

In [ ]:
# List fields, columns, and structure of each record set
for record_set in dataset.metadata.record_sets:
    rs_id = record_set['@id']
    print(f"\nRecord Set @id: {rs_id}")
    print(f"  Name: {record_set.get('name', '<unnamed>')}")
    if 'fields' in record_set:
        print("  Fields:")
        for field in record_set['fields']:
            print(f"    - Field @id: {field['@id']}, name: {field.get('name', '<unnamed>')}, dataType: {field.get('dataType', '<unknown>')}")
    if 'columns' in record_set:
        print("  Columns:")
        for column in record_set['columns']:
            print(f"    - Column @id: {column['@id']}, name: {column.get('name', '<unnamed>')}")

# Show a preview record for each record set - by @id
for record_set in dataset.metadata.record_sets:
    rs_id = record_set['@id']
    print(f"\nPreview record from Record Set @id: {rs_id}")
    ds_records = dataset.records(record_set=rs_id)
    try:
        print(next(ds_records))
    except StopIteration:
        print("  [No records found]")

## 3. Data Extraction
Load data from specific record sets into a DataFrame for analysis. All record set and field references are by their `@id`.

In [ ]:
# Gather all record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records_iterator = dataset.records(record_set=record_set_id)
    records_list = list(records_iterator)
    df = pd.DataFrame(records_list)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for Record Set @id: {record_set_id}")
    print(f"  Columns (@id): {df.columns.tolist()}")
    display(df.head())
    print(f"  Shape: {df.shape}")

# Select primary record set (based on number of columns or records; here we pick the first for demonstration)
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nSelected main Record Set @id: {main_record_set_id}")
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric columns, categorize or group data. All fields referenced by their `@id`.

In [ ]:
# Identify a numeric field (column) by inspecting columns
numeric_field_id = None

# Try to heuristically pick a numeric field
for col in main_df.columns:
    if main_df[col].dtype in ['float64', 'int64']:
        numeric_field_id = col
        break
# If no numeric, try to convert one
if numeric_field_id is None:
    for col in main_df.columns:
        # Try to convert string columns to numeric, skip errors
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
            if main_df[col].notna().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

print(f"Selected numeric field for EDA: {numeric_field_id}")

if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notna().sum() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Identify a categorical/grouping field (prefer those with few unique values)
    group_field_id = None
    for col in main_df.columns:
        if col == numeric_field_id:
            continue
        nunique = main_df[col].nunique()
        if nunique > 1 and nunique < 30:
            group_field_id = col
            break

    print(f"Selected group field for grouping: {group_field_id}")
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if numeric_field_id is not None and group_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and process a Croissant dataset using the `mlcroissant` library.
We explored available record set and field `@id`s, loaded raw data, performed exploratory data analysis (including normalization and grouping by attributes), and generated visualizations. For more advanced analysis, users can continue building upon this notebook by referencing entities by their unique Croissant `@id`s.